In [72]:
import graph_tool.all as gt
import numpy as np
from graph_tool.topology import extract_largest_component
import matplotlib.pyplot as plt
import os
import time
import pandas as pd

In [2]:
analysis_folder = "./analysis_data/"

In [11]:
def is_weight_type_supported(type):
    supported_types = ['int16_t', 'int32_t', 'int64_t', 'float', 'double', 'long double']
    return type in supported_types

In [12]:
def prepare_folder(graph_name):
    safe_graph_name = graph_name.replace('/', '__')
    path = f"{analysis_folder}{safe_graph_name}"
    if not os.path.exists(path):
        os.makedirs(path)
    return path

In [13]:
def get_mst(graph, edge_property = np.nan):
    if(edge_property is np.nan):
        weight_key = "weight"
    else:
        weight_key = edge_property
    
    if weight_key in graph.edge_properties and is_weight_type_supported(graph.ep[weight_key].value_type()):
        graph.ep[weight_key].a *= -1
    else:
        # TODO: add logic of throw error
        print("no weight")
        return False
    mst_tree = gt.min_spanning_tree(graph, graph.ep[weight_key])

    return gt.GraphView(graph, efilt=mst_tree)

In [14]:
def has_parallel_edges(g: gt.Graph) -> bool:
    if g.num_edges() == 0:
        return False

    edge_counts = {}
    for e in g.edges():
        s_idx = int(e.source())
        t_idx = int(e.target())
        
        pair = (s_idx, t_idx)
        if not g.is_directed():
            pair = tuple(sorted((s_idx, t_idx)))
        
        current_count = edge_counts.get(pair, 0) + 1
        if current_count > 1:
            return True
        edge_counts[pair] = current_count
        
    return False

In [15]:
def simplify_graph_with_weights(graph: gt.Graph, weight_name = np.nan) -> tuple[gt.Graph, gt.EdgePropertyMap]:
    processed_weight_map: gt.EdgePropertyMap
    g = graph.copy()
    if isinstance(weight_name, str):
        # Scenario 1: Sum weights from an existing property.
        if weight_name not in g.ep:
            raise ValueError(f"Edge property '{weight_name}' not found in the graph. Cannot sum non-existent weights.")
        
        # `contract_parallel_edges` uses weights from `g.ep[weight_name]` before `g` is modified.
        existing_weights_to_sum = g.ep[weight_name] 
        
        # `g` is modified in-place. `summed_map` has summed weights for the new edges.
        summed_map = gt.contract_parallel_edges(g, existing_weights_to_sum)
        
        # Update the specified edge property with the summed weights.
        g.ep[weight_name] = summed_map
        processed_weight_map = summed_map

    elif weight_name is np.nan:
        # Scenario 2: Default - count parallel edges, store in "weight" property.
        # `g` is modified in-place. `multiplicity_map` stores edge counts.
        if 'weight' in g.ep and is_weight_type_supported(graph.ep['weight'].value_type()):
            existing_weights_to_sum = g.ep['weight']
            summed_map = gt.contract_parallel_edges(g, existing_weights_to_sum)

        else:
            summed_map = gt.contract_parallel_edges(g)
        
        # Store counts in `g.ep["weight"]` (overwrites if "weight" exists).
        g.ep["weight"] = summed_map 
        processed_weight_map = summed_map
    else:
        raise TypeError(f"weight_name must be a string or np.nan, not {type(weight_name)}")

    return g, processed_weight_map

In [16]:
def directed_to_undirected_sum_weights(
    g_directed: gt.Graph, 
    existing_weight_name = np.nan, 
    new_undirected_weight_name: str = "weight"
) -> tuple[gt.Graph, gt.EdgePropertyMap]:
    if not g_directed.is_directed():
        raise ValueError("Input graph must be directed.")

    g_work = g_directed.copy()

    _, simplified_weights_prop_map_in_g_work = simplify_graph_with_weights(g_work, weight_name=existing_weight_name)
    
    intermediate_weight_prop_name = existing_weight_name if isinstance(existing_weight_name, str) else "weight"
    
    if intermediate_weight_prop_name not in g_work.ep:
        raise RuntimeError(f"Internal error: Weight property '{intermediate_weight_prop_name}' not found in working graph after simplification.")

    g_undirected = gt.Graph(directed=False)
    g_undirected.add_vertex(n=g_work.num_vertices()) 

    final_undirected_weights_map = g_undirected.new_edge_property(simplified_weights_prop_map_in_g_work.value_type())
    final_undirected_weight_name = existing_weight_name if isinstance(existing_weight_name, str) else new_undirected_weight_name
    g_undirected.ep[final_undirected_weight_name] = final_undirected_weights_map

    undirected_edge_weights_aggregated = {} 

    for edge in g_work.edges():
        u = edge.source()
        v = edge.target()
        u_idx = g_work.vertex_index[u]
        v_idx = g_work.vertex_index[v]
        
        weight = g_work.ep[intermediate_weight_prop_name][edge]

        undir_pair = tuple(sorted((u_idx, v_idx)))
        
        undirected_edge_weights_aggregated[undir_pair] = \
            undirected_edge_weights_aggregated.get(undir_pair, 0) + weight

    for (idx1, idx2), total_weight in undirected_edge_weights_aggregated.items():
        vertex1_undir = g_undirected.vertex(idx1)
        vertex2_undir = g_undirected.vertex(idx2)
        
        new_edge_undir = g_undirected.add_edge(vertex1_undir, vertex2_undir)
        final_undirected_weights_map[new_edge_undir] = total_weight
        
    return g_undirected, final_undirected_weights_map

In [17]:
def start_drawing(graph_name, weight_key=np.nan):
    print(f"Starting to draw: {graph_name}")
    path = prepare_folder(graph_name)

    graph = gt.collection.ns[graph_name]
    if (has_parallel_edges(graph)):
        graph, _ = simplify_graph_with_weights(graph, weight_key)

    if (graph.is_directed()):
        graph, _ = directed_to_undirected_sum_weights(graph, weight_key)

    print(f"Drawing largest component: {graph_name}")
    largest_component = extract_largest_component(
        graph, directed=False, prune=True)
    
    gt.graph_draw(largest_component, output=f'{path}/largest_component.png')
    
    print(f"Getting MST: {graph_name}")
    mst = get_mst(largest_component, weight_key)
    if isinstance(mst, gt.GraphView):
        print(f"Drawing MST: {graph_name}")
        gt.graph_draw(mst, output=f'{path}/mst_draw.png')

In [18]:
def start_drawing_pipe(data):
    print("Computing results...")
    start_time = time.time()

    for graph_data in data:
        if(isinstance(graph_data,tuple) and len(graph_data) >= 2):
            start_drawing(graph_data[0], graph_data[1])
        else:
            start_drawing(graph_data)
    
    end_time = time.time()

    print(f"Computation finished in {end_time - start_time:.2f} seconds")


In [ ]:
results = start_drawing_pipe(
    [
        # 'new_zealand_collab' # Test 
        'word_assoc',
        'human_brains/BNU1_0025915_2_DTI_DS16784',
        ('us_agencies/aggregate', 'link_counts'),
        'physics_collab/arXiv',
        # 'libimseti',
        # 'lkml_reply',
        # 'topology',
        # ('us_roads/DE', 'distance')
    ]
)

Computing results...
Starting to draw: word_assoc
Drawing largest component: word_assoc
Getting MST: word_assoc
Drawing MST: word_assoc
Starting to draw: human_brains/BNU1_0025915_2_DTI_DS16784
Drawing largest component: human_brains/BNU1_0025915_2_DTI_DS16784
Getting MST: human_brains/BNU1_0025915_2_DTI_DS16784
Drawing MST: human_brains/BNU1_0025915_2_DTI_DS16784
Starting to draw: us_agencies/aggregate
Drawing largest component: us_agencies/aggregate
Getting MST: us_agencies/aggregate
no weight
Starting to draw: physics_collab/arXiv
Drawing largest component: physics_collab/arXiv
Getting MST: physics_collab/arXiv
Drawing MST: physics_collab/arXiv
Computation finished in 640.79 seconds


In [16]:
results = start_drawing_pipe(
    [
        ('us_agencies/aggregate', 'link_counts'),
    ]
)

Computing results...
Starting to draw: us_agencies/aggregate
Drawing largest component: us_agencies/aggregate
Getting MST: us_agencies/aggregate
Drawing MST: us_agencies/aggregate
Computation finished in 435.31 seconds


# Description pipeline:

In [57]:
def nan_dict(name, is_mst):
    return {
        'name' : name,
        'is_mst' : is_mst,
        'vert_num' : np.nan,
        'edge_num' : np.nan,
        'density' : np.nan,
        'avg_degree' : np.nan
    }

In [78]:
def calc_metrics(graph_name, graph, is_mst):
    edge_num = graph.num_edges()
    node_num = graph.num_vertices()

    V = graph.num_vertices()
    E = graph.num_edges()

    graph_density = (2 * E) / (V * (V - 1) / 2)

    degrees = graph.degree_property_map("total").fa

    avg_degree_centrality = np.mean(degrees)

    return {
        'name' : graph_name,
        'is_mst' : is_mst,
        'vert_num' : node_num,
        'edge_num' : edge_num,
        'density' : graph_density,
        'avg_degree' : avg_degree_centrality
    }
    

In [70]:
def get_metrics_result(graph_name, weight_key=np.nan):
    print(f"Started metrics for: {graph_name}")

    graph = gt.collection.ns[graph_name]
    if (has_parallel_edges(graph)):
        graph, _ = simplify_graph_with_weights(graph, weight_key)

    if (graph.is_directed()):
        graph, _ = directed_to_undirected_sum_weights(graph, weight_key)

    print(f"Preparing LC: {graph_name}")
    largest_component = extract_largest_component(
        graph, directed=False, prune=True)
    
    lc_result = calc_metrics(graph_name, largest_component, False)
    
    print(f"Getting MST: {graph_name}")
    mst = get_mst(largest_component, weight_key)
    mst_result = nan_dict(graph_name, True)
    if isinstance(mst, gt.GraphView):
        print(f"Preparing MST: {graph_name}")
        mst_result = calc_metrics(graph_name, mst, True)

    return lc_result, mst_result


In [64]:
def start_metrics_pipe(data):
    results = []
    print("Computing results...")
    start_time = time.time()

    for graph_data in data:
        if(isinstance(graph_data,tuple) and len(graph_data) >= 2):
            graph_result, mst_result = get_metrics_result(graph_data[0], graph_data[1])
        else:
            graph_result, mst_result = get_metrics_result(graph_data)
        
        results.append(graph_result)
        results.append(mst_result)

    end_time = time.time()

    print(f"Computation finished in {end_time - start_time:.2f} seconds")

    return results

In [79]:
metrics_results = start_metrics_pipe(
    [
        'word_assoc',
        'human_brains/BNU1_0025915_2_DTI_DS16784',
        ('us_agencies/aggregate', 'link_counts'),
        'physics_collab/arXiv',
        'topology',
        ('us_roads/DE', 'distance')
    ]
)

Computing results...
Started metrics for: word_assoc
Preparing LC: word_assoc
Getting MST: word_assoc
Preparing MST: word_assoc
Started metrics for: human_brains/BNU1_0025915_2_DTI_DS16784
Preparing LC: human_brains/BNU1_0025915_2_DTI_DS16784
Getting MST: human_brains/BNU1_0025915_2_DTI_DS16784
Preparing MST: human_brains/BNU1_0025915_2_DTI_DS16784
Started metrics for: us_agencies/aggregate
Preparing LC: us_agencies/aggregate
Getting MST: us_agencies/aggregate
Preparing MST: us_agencies/aggregate
Started metrics for: physics_collab/arXiv
Preparing LC: physics_collab/arXiv
Getting MST: physics_collab/arXiv
Preparing MST: physics_collab/arXiv
Started metrics for: topology
Preparing LC: topology
Getting MST: topology
Preparing MST: topology
Started metrics for: us_roads/DE
Preparing LC: us_roads/DE
Getting MST: us_roads/DE
Preparing MST: us_roads/DE
Computation finished in 25.41 seconds


In [80]:
metrics_result_df = pd.DataFrame(metrics_results)

In [81]:
metrics_result_df

,name,is_mst,vert_num,edge_num,density,avg_degree
0,word_assoc,False,23132,297646,0.002225,25.734566833823276
1,word_assoc,True,23132,23131,0.000173,1.9999135396852845
2,human_brains/BNU1_0025915_2_DTI_DS16784,False,12454,360366,0.009294,57.87152722017023
3,human_brains/BNU1_0025915_2_DTI_DS16784,True,12454,12453,0.000321,1.9998394090252127
4,us_agencies/aggregate,False,42800,451769,0.000987,21.11070093457944
5,us_agencies/aggregate,True,42800,42799,0.000093,1.9999532710280374
6,physics_collab/arXiv,False,8798,27416,0.001417,6.232325528529211
7,physics_collab/arXiv,True,8798,8797,0.000455,1.9997726756080927
8,topology,False,34761,107720,0.000357,6.197750352406432
9,topology,True,34761,34760,0.000115,1.999942464255919
